In [1]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm

In [ ]:
ABS = [
    ("CRAN",          {185180, 1001}),
    ("CREATIS",       {139739}),
    ("CRIL",          {90448, 1628}),
    ("CRISTAL",       {410272, 389110, 388977, 390300, 183073, 111636, 24885, 2546, 186929}),
    ("DI ENS",        {25027}),
    ("CROSSING (IRL)",{1063106}),
    ("ETIS",          {1003474, 1061575, 1087906, 1003348}),
    ("FILOFOCS (UMI)",{1006288, 1006289}),
    ("GIPSA-Lab",     {1043333, 1042376, 24470}),
    ("GREYC",         {150}),
    ("G-SCOP",        {1043137, 1041927, 74240}),
    ("HEUDIASYC",     {389870}),
    ("I3S",           {13009, 552896, 1079434}),
    ("ICUBE",         {217648, 1073080}),
    ("IDRIS",         {1823}),
    ("IPAL (IRL)",    {542003, 220880, 138926}),
    ("IRIF",          {1005016, 444497}),
    ("IRISA",         {491183, 491231, 490899, 491188, 1092618, 491177, 1092619, 490900,
                       419364, 419370, 105128, 2494, 25255, 419365, 419367, 491230,
                       419363, 419366, 491232, 419362, 1099404, 545024, 1099406,
                       1099401, 1099402, 1099435, 525233, 1088566, 1088569, 495900,
                       489780, 1092631, 1092630, 1092628, 1092632, 1092626, 1092625, 1092629}),
    ("IRIT",          {34499, 1082335}),
    ("ISIR",          {541937, 96164}),
    ("JFLI",          {542009, 229050}),
    ("L2S",           {1051117, 1289}),
    ("LAAS",          {459}),
    ("LABRI",         {3102}),
    ("LAB-STICC",     {486345, 491660, 199324, 81533, 1089048}),
    ("LAMIH",         {1067790, 1303}),
    ("LAMSADE",       {989}),
    ("LIG",           {1043301, 1041964, 24471}),
    ("LIGM",          {1001627, 3210}),
    ("LIMOS",         {1063677, 490706, 857}),
    ("LIP",           {35418}),
    ("LIP6",          {541703, 233, 1095103}),
    ("LIPN",          {1000994, 994, 1086916, 1056718}),
    ("LIRIS",         {2003, 1086665}),
    ("LIRMM",         {181, 1071941}),
    ("LIS",           {527033, 199402, 199394, 862, 178374}),
    ("LISN",          {1061259, 1041968, 247329, 2544, 1050003, 81750}),
    ("LIX",           {2071, 1041697, 1071530, 1070048}),
    ("LMF",           {1065710, 1042689, 2571}),
    ("LORIA",         {206040, 466633}),
    ("LS2N",          {1088564, 473973, 95421, 1693, 21439}),
    ("MDLS",          {210816}),
    ("RELAX",         {1040410, 528907}),
    ("STMS",          {541779, 1374}),
    ("TIMA",          {1043043, 1044023, 640}),
    ("TIMC IMAG",     {1043049, 1070489, 1042061, 707, 574002, 555959, 1056575}),
    ("VERIMAG",       {1043148, 1041816, 194}),
]

## Recherche des identifiants des labos sur HAL (ROR, RNSR etc.)

Dans l'index Solr de api.hal.science/ref/structure, chaque structure est indexée comme un "document Solr". HAL a choisi d'utiliser l'identifiant numérique de la structure comme clé primaire Solr (docid). Il n'existe donc pas de champ séparé structId_i dans cet index — la valeur 150 (structId du GREYC) est le docid de ce document dans l'index.

C'est un choix d'implémentation de HAL : dans l'index /ref/structure, docid = identifiant de structure. Ce n'est pas une identité conceptuelle, c'est juste la clé primaire Solr qui a été alimentée avec les structIds.

La réponse complète révèle aussi un champ intéressant : aliasDocid_i contient les anciens identifiants d'une structure (fusions, renommages). C'est probablement pourquoi certains labos dans ABS ont plusieurs structIds — ce sont d'anciennes versions fusionnées dans la structure courante.

In [4]:
HAL_API = "https://api.hal.science/ref/structure/"
FIELDS = ["docid", "label_s", "acronym_s", "aliasDocid_i",
          "idref_s", "ror_s", "rnsr_s", "wikidata_s", "isni_s"]

def fetch_ids_for_struct(struct_ids):
    """Interroge l'API HAL pour un ensemble de structIds (= docid) et retourne les identifiants externes."""
    q = " OR ".join(f"docid:{sid}" for sid in struct_ids)
    params = {
        "q":   q,
        "fl":  ",".join(FIELDS),
        "rows": 200,
        "wt":  "json",
    }
    resp = requests.get(HAL_API, params=params)
    resp.raise_for_status()
    return resp.json().get("response", {}).get("docs", [])

rows = []
for labo, struct_ids in tqdm(ABS, desc="Labos"):
    docs = fetch_ids_for_struct(struct_ids)
    time.sleep(0.1)

    def collect(field):
        vals = set()
        for d in docs:
            v = d.get(field, [])
            vals.update(v if isinstance(v, list) else [v])
        return sorted(vals)

    rows.append({
        "labo":         labo,
        "structIds":    sorted(struct_ids),
        "aliasDocids":  collect("aliasDocid_i"),
        "idref":        collect("idref_s"),
        "isni":         collect("isni_s"),
        "ror":          collect("ror_s"),
        "rnsr":         collect("rnsr_s"),
        "wikidata":     collect("wikidata_s"),
    })

df_ids = pd.DataFrame(rows)
df_ids


Labos:   0%|          | 0/50 [00:00<?, ?it/s]

,labo,structIds,aliasDocids,idref,isni,ror,rnsr,wikidata
0,CMP,[244423],[],[],[],[],[],[]
1,CRAN,"[1001, 185180]","[191005, 221880, 388369, 389041, 389738, 39049...",[03500472X],[0000000121518763],[https://ror.org/022r5hc56],[200112440X],[]
2,CREATIS,[139739],"[193461, 403068, 403069, 437584, 514895, 51937...",[19513270X],[0000000406380358],[https://ror.org/03smk3872],[200717526Z],[]
3,CRIL,"[1628, 90448]","[233545, 400222, 566931, 1095133]",[08345585X],[],[],[200812835W],[]
4,CRISTAL,"[2546, 24885, 111636, 183073, 186929, 388977, ...","[9316, 39658, 221912, 244503, 246178, 264592, ...",[18388695X],[],[https://ror.org/05vrs3189],[201521249L],[Q116959497]
5,DI ENS,[25027],"[219326, 240109, 417709, 1191418]",[148034055],[],[],[199812876J],[]
6,CROSSING (IRL),[1063106],[],[],[],[],[],[]
7,ETIS,"[1003348, 1003474, 1061575, 1087906]","[1210, 127522, 265775, 397529, 400182, 430079,...",[129737623],[],[https://ror.org/0592ata02],[200212719W],[]
8,FILOFOCS (UMI),"[1006288, 1006289]",[],[],[],[],[201922944L],[]
9,GIPSA-Lab,"[24470, 1042376, 1043333]","[300246, 391449, 395713, 414384, 414416, 41466...",[137251122],[0000000118823396],[https://ror.org/02wrme198],[200711885T],[]


In [5]:
# Export CSV — les listes sont sérialisées séparées par " | "
df_csv = df_ids.copy()
for col in ["structIds", "aliasDocids", "idref", "isni", "ror", "rnsr", "wikidata"]:
    df_csv[col] = df_csv[col].apply(lambda x: " | ".join(str(v) for v in x))

df_csv.to_csv("labo_identifiants.csv", index=False, encoding="utf-8-sig")
print("Fichier labo_identifiants.csv enregistré.")
df_csv


Fichier labo_identifiants.csv enregistré.


,labo,structIds,aliasDocids,idref,isni,ror,rnsr,wikidata
0,CMP,244423,,,,,,
1,CRAN,1001 | 185180,191005 | 221880 | 388369 | 389041 | 389738 | 3...,03500472X,0000000121518763,https://ror.org/022r5hc56,200112440X,
2,CREATIS,139739,193461 | 403068 | 403069 | 437584 | 514895 | 5...,19513270X,0000000406380358,https://ror.org/03smk3872,200717526Z,
3,CRIL,1628 | 90448,233545 | 400222 | 566931 | 1095133,08345585X,,,200812835W,
4,CRISTAL,2546 | 24885 | 111636 | 183073 | 186929 | 3889...,9316 | 39658 | 221912 | 244503 | 246178 | 2645...,18388695X,,https://ror.org/05vrs3189,201521249L,Q116959497
5,DI ENS,25027,219326 | 240109 | 417709 | 1191418,148034055,,,199812876J,
6,CROSSING (IRL),1063106,,,,,,
7,ETIS,1003348 | 1003474 | 1061575 | 1087906,1210 | 127522 | 265775 | 397529 | 400182 | 430...,129737623,,https://ror.org/0592ata02,200212719W,
8,FILOFOCS (UMI),1006288 | 1006289,,,,,201922944L,
9,GIPSA-Lab,24470 | 1042376 | 1043333,300246 | 391449 | 395713 | 414384 | 414416 | 4...,137251122,0000000118823396,https://ror.org/02wrme198,200711885T,


In [ ]:
SUSPECT_THRESHOLD = 2  # numFound ≤ ce seuil → on soupçonne un alias non résolu

def resolve_valid_struct_ids(struct_ids, year=YEAR):
    """
    Pour chaque structId, teste empiriquement si la requête /search/ retourne
    des publications pour l'année donnée.

    Si numFound <= SUSPECT_THRESHOLD : l'alias n'est probablement pas résolu dans
    l'index → on cherche dans ref/structure la structure VALID ayant le même acronyme
    et on la substitue.

    Si numFound > SUSPECT_THRESHOLD : l'alias fonctionne (ou c'est vraiment un petit
    labo) → on conserve l'ID original.

    Retourne :
      resolved   : set de structIds à utiliser dans /search/
      accepted   : set de tous les structIds à accepter dans le filtre de facette
                   (orig + valid, pour couvrir les deux encodages possibles)
    """
    resolved = set()
    accepted = set()

    for sid in struct_ids:
        # Test empirique
        params = {
            "q":    f"structId_i:{sid} AND producedDateY_i:{year}",
            "rows": 0,
            "wt":   "json",
        }
        r = requests.get(HAL_SEARCH, params=params)
        r.raise_for_status()
        num_found = r.json().get("response", {}).get("numFound", 0)
        time.sleep(0.05)

        if num_found > SUSPECT_THRESHOLD:
            # L'ID fonctionne → on le garde
            resolved.add(sid)
            accepted.add(sid)
        else:
            # Suspect → on cherche la structure VALID par acronyme
            p2 = {"q": f"docid:{sid}", "fl": "docid,valid_s,acronym_s", "rows": 1, "wt": "json"}
            docs = requests.get(HAL_API, params=p2).json().get("response", {}).get("docs", [])
            time.sleep(0.05)

            acronym = docs[0].get("acronym_s", "") if docs else ""
            if acronym:
                p3 = {"q": f'acronym_s:"{acronym}" AND valid_s:VALID',
                      "fl": "docid,acronym_s", "rows": 10, "wt": "json"}
                valid_docs = requests.get(HAL_API, params=p3).json().get("response", {}).get("docs", [])
                time.sleep(0.05)

                if valid_docs:
                    for vd in valid_docs:
                        vid = int(vd["docid"])
                        resolved.add(vid)
                        accepted.update([sid, vid])   # accepter les deux dans la facette
                else:
                    # Aucune structure VALID trouvée → on garde l'original
                    resolved.add(sid)
                    accepted.add(sid)
            else:
                resolved.add(sid)
                accepted.add(sid)

    return resolved, accepted


# --- Résolution pour tous les labos ---
print(f"Résolution empirique des structIds (seuil numFound ≤ {SUSPECT_THRESHOLD}) :")
resolved_ABS = []
for labo, struct_ids in tqdm(ABS, desc="Résolution"):
    valid_ids, accepted_ids = resolve_valid_struct_ids(struct_ids)
    resolved_ABS.append((labo, struct_ids, valid_ids, accepted_ids))

    substitutions = valid_ids - set(struct_ids)
    suppressions  = set(struct_ids) - valid_ids
    if substitutions or suppressions:
        print(f"  {labo}: retire {suppressions}, ajoute {substitutions}")

print("Résolution terminée.")


In [21]:
HAL_SEARCH = "https://api.hal.science/search/"
HAL_AUTHOR = "https://api.hal.science/ref/author/"
YEAR = 2024

def get_authors_for_lab(valid_ids, accepted_ids, year=YEAR):
    """
    Retourne un dict {authId: authName} pour les auteurs affiliés à ce labo.

    valid_ids   : structIds VALIDES à utiliser dans la requête /search/
    accepted_ids: ensemble de tous les structIds à accepter dans le filtre de facette
                  (inclut les OLD et les VALID pour couvrir les deux encodages possibles)
    """
    struct_query = " OR ".join(str(sid) for sid in valid_ids)
    params = {
        "q":              f"structId_i:({struct_query}) AND producedDateY_i:{year}",
        "rows":           0,
        "facet":          "true",
        "facet.field":    "structHasAuthId_fs",
        "facet.limit":    -1,
        "facet.mincount": 1,
        "wt":             "json",
    }
    resp = requests.get(HAL_SEARCH, params=params)
    resp.raise_for_status()
    facet_list = (resp.json()
                  .get("facet_counts", {})
                  .get("facet_fields", {})
                  .get("structHasAuthId_fs", []))

    accepted_str = {str(sid) for sid in accepted_ids}
    authors = {}
    for val in facet_list[::2]:
        joinsep = val.split("_JoinSep_")
        if len(joinsep) != 2:
            continue
        struct_part, auth_part = joinsep
        struct_id = struct_part.split("_FacetSep_")[0]
        if struct_id not in accepted_str:
            continue
        auth_parts = auth_part.split("_FacetSep_")
        auth_id   = auth_parts[0]
        auth_name = auth_parts[1] if len(auth_parts) > 1 else ""
        authors[auth_id] = auth_name
    return authors


# --- Collecte des auteurs par labo (avec structIds résolus) ---
lab_authors = {}
all_auth_ids = set()

for labo, orig_ids, valid_ids, valid_to_orig in tqdm(resolved_ABS, desc="Auteurs par labo"):
    # Tous les IDs acceptables dans la facette = union des orig + valid
    accepted_ids = set(orig_ids) | valid_ids
    authors = get_authors_for_lab(valid_ids, accepted_ids)
    lab_authors[labo] = authors
    all_auth_ids.update(authors.keys())
    time.sleep(0.1)

print(f"{len(all_auth_ids)} auteurs distincts trouvés (toutes structures confondues).")


Auteurs par labo:   0%|          | 0/47 [00:00<?, ?it/s]

12192 auteurs distincts trouvés (toutes structures confondues).


In [22]:
def batch_get_author_info(auth_ids, batch_size=50):
    """
    Interroge ref/author par lots de batch_size.
    Retourne un dict authId -> {"idHal": str, "orcids": list[str]}
    Le champ orcidId_s contient des URLs complètes (https://orcid.org/...).
    """
    auth_ids = list(auth_ids)
    result = {}
    for i in tqdm(range(0, len(auth_ids), batch_size), desc="Infos auteurs"):
        batch = auth_ids[i:i + batch_size]
        q = " OR ".join(f"docid:{aid}" for aid in batch)
        params = {
            "q":    q,
            "fl":   "docid,idHal_s,orcidId_s",
            "rows": len(batch) + 5,
            "wt":   "json",
        }
        resp = requests.get(HAL_AUTHOR, params=params)
        resp.raise_for_status()
        for d in resp.json().get("response", {}).get("docs", []):
            orcids = d.get("orcidId_s", [])
            result[d["docid"]] = {
                "idHal":  d.get("idHal_s", ""),
                "orcids": orcids if isinstance(orcids, list) else [orcids],
            }
        time.sleep(0.1)
    return result


author_cache = batch_get_author_info(all_auth_ids)
print(f"Infos récupérées pour {len(author_cache)} / {len(all_auth_ids)} auteurs.")


Infos auteurs:   0%|          | 0/244 [00:00<?, ?it/s]

Infos récupérées pour 12188 / 12192 auteurs.


In [23]:
ORCID_API = "https://pub.orcid.org/v3.0"

def get_orcid_employment(orcid_url):
    """
    Retourne la liste des emplois *courants* (end-date absente) déclarés sur ORCID.
    Chaque emploi est un dict {"role": str, "org": str}.
    orcid_url : "https://orcid.org/XXXX-XXXX-XXXX-XXXX"
    """
    orcid_id = orcid_url.rstrip("/").split("/")[-1]
    try:
        resp = requests.get(
            f"{ORCID_API}/{orcid_id}/employments",
            headers={"Accept": "application/json"},
            timeout=10,
        )
        if resp.status_code != 200:
            return []
        current = []
        for group in resp.json().get("affiliation-group", []):
            for wrapper in group.get("summaries", []):
                emp = wrapper.get("employment-summary", {})
                if emp.get("end-date") is not None:
                    continue                          # poste terminé → ignoré
                role = (emp.get("role-title") or "").strip()
                org  = ((emp.get("organization") or {}).get("name") or "").strip()
                if role or org:
                    current.append({"role": role, "org": org})
        return current
    except Exception:
        return []


# Collecter tous les ORCIDs uniques présents dans author_cache
all_orcids = {
    orcid
    for info in author_cache.values()
    for orcid in info.get("orcids", [])
}
print(f"{len(all_orcids)} ORCIDs uniques à interroger sur ORCID.org…")

# orcid_url -> liste d'emplois courants
orcid_employment = {}
for orcid_url in tqdm(sorted(all_orcids), desc="Emplois ORCID"):
    orcid_employment[orcid_url] = get_orcid_employment(orcid_url)
    time.sleep(0.15)   # API publique ORCID : ~6 req/s sans token

n_avec = sum(1 for v in orcid_employment.values() if v)
print(f"Emploi(s) courant(s) trouvé(s) pour {n_avec} / {len(all_orcids)} ORCIDs.")


5696 ORCIDs uniques à interroger sur ORCID.org…


Emplois ORCID:   0%|          | 0/5696 [00:00<?, ?it/s]

Emploi(s) courant(s) trouvé(s) pour 3432 / 5696 ORCIDs.


In [24]:
def format_employment(orcids):
    """
    Agrège les emplois courants de tous les ORCIDs d'un auteur.
    Retourne deux chaînes : roles et employeurs (séparés par ' | ' si multiples).
    """
    roles, orgs = [], []
    seen = set()
    for orcid_url in orcids:
        for emp in orcid_employment.get(orcid_url, []):
            key = (emp["role"], emp["org"])
            if key not in seen:
                seen.add(key)
                if emp["role"]: roles.append(emp["role"])
                if emp["org"]:  orgs.append(emp["org"])
    return " | ".join(roles), " | ".join(orgs)


rows = []
for labo, struct_ids in ABS:
    for auth_id, auth_name in sorted(lab_authors[labo].items(), key=lambda x: x[1]):
        info   = author_cache.get(auth_id, {})
        orcids = info.get("orcids", [])
        role, employeur = format_employment(orcids)
        rows.append({
            "labo":      labo,
            "authId":    auth_id,
            "nom":       auth_name,
            "idHAL":     info.get("idHal", ""),
            "orcid":     " | ".join(orcids),
            "role":      role,      # texte libre ORCID (ex: "Full professor", "PhD student"…)
            "employeur": employeur, # organisation déclarée sur ORCID
        })

df_authors = pd.DataFrame(rows)

df_authors.to_csv("labo_auteurs_2024.csv", index=False, encoding="utf-8-sig")
print(f"{len(df_authors)} lignes exportées dans labo_auteurs_2024.csv")
print(f"Emploi ORCID renseigné pour {(df_authors['role'] != '').sum()} auteurs "
      f"({100*(df_authors['role'] != '').mean():.0f}%).")
df_authors


12547 lignes exportées dans labo_auteurs_2024.csv
Emploi ORCID renseigné pour 2710 auteurs (22%).


,labo,authId,nom,idHAL,orcid,role,employeur
0,CRAN,2893315-1282086,Abir Bouaouda,,,,
1,CRAN,3657463-0,Abir Ismaili-Aoui,,,,
2,CRAN,170448-1502602,Agnès Leroux,,https://orcid.org/0000-0001-5222-1954,MD,Institut de Cancérologie de Lorraine
3,CRAN,2718656-1225099,Ahmed Zghal,,,,
4,CRAN,53845-746881,Alaaeddine Chaoub,alaaeddine-chaoub,,,
...,...,...,...,...,...,...,...
12542,VERIMAG,4664-0,Thao Dang,,,,
12543,VERIMAG,4664-180504,Thao Dang,thao-dang,https://orcid.org/0000-0002-3637-1415,Directrice de recherche,Université Grenoble Alpes | CNRS Délégation Rh...
12544,VERIMAG,2580952-1162762,Thomas Vigouroux,thomas-vigouroux,https://orcid.org/0000-0001-6396-0285,PhD student,Verimag
12545,VERIMAG,2580952-1550591,Thomas Vigouroux,,,,


In [25]:
def stats_orcid(df):
    """Calcule les statistiques ORCID sur un sous-ensemble du dataframe."""
    n_total    = len(df)
    has_orcid  = df["orcid"] != ""
    n_orcid    = has_orcid.sum()
    n_no_orcid = (~has_orcid).sum()
    pct_orcid  = 100 * n_orcid / n_total if n_total else 0
    n_multi    = (df["orcid"].str.count(r"\|") >= 1).sum()

    # % par employeur : n_auteurs_avec_cet_employeur / n_total_auteurs_du_labo
    # (tous ont un ORCID puisque l'employeur vient d'ORCID)
    emp_stats = {}
    df_emp = df[df["employeur"] != ""].copy()
    df_emp = df_emp.assign(employeur=df_emp["employeur"].str.split(r" \| ")).explode("employeur")
    df_emp["employeur"] = df_emp["employeur"].str.strip()
    for emp, grp in df_emp.groupby("employeur"):
        n_e = len(grp)
        emp_stats[emp] = f"{n_e}/{n_total} ({100*n_e/n_total:.0f}%)"

    # % par fonction : n_auteurs_avec_cette_fonction / n_total_auteurs_du_labo
    role_stats = {}
    df_role = df[df["role"] != ""].copy()
    df_role = df_role.assign(role=df_role["role"].str.split(r" \| ")).explode("role")
    df_role["role"] = df_role["role"].str.strip()
    for role, grp in df_role.groupby("role"):
        n_r = len(grp)
        role_stats[role] = f"{n_r}/{n_total} ({100*n_r/n_total:.0f}%)"

    return {
        "n_auteurs":           n_total,
        "n_avec_orcid":        n_orcid,
        "pct_avec_orcid":      round(pct_orcid, 1),
        "n_multi_orcid":       n_multi,
        "n_sans_orcid":        n_no_orcid,
        "orcid_par_employeur": emp_stats,
        "orcid_par_fonction":  role_stats,
    }


# --- Stats par labo ---
rows_stats = []
for labo in df_authors["labo"].unique():
    df_labo = df_authors[df_authors["labo"] == labo]
    s = stats_orcid(df_labo)
    rows_stats.append({"labo": labo, **s})

# --- Stats globales : dédupliquer les authIds ---
df_unique_authors = df_authors.drop_duplicates(subset="authId")
s_global = stats_orcid(df_unique_authors)
rows_stats.append({"labo": "TOUS LES LABOS", **s_global})

df_stats = pd.DataFrame(rows_stats).set_index("labo")

print("=== Vue d'ensemble ===")
display(df_stats[["n_auteurs", "n_avec_orcid", "pct_avec_orcid", "n_multi_orcid", "n_sans_orcid"]])


=== Vue d'ensemble ===


,n_auteurs,n_avec_orcid,pct_avec_orcid,n_multi_orcid,n_sans_orcid
labo,,,,,
CRAN,249,172,69.1,1,77
CREATIS,286,147,51.4,2,139
CRIL,81,41,50.6,0,40
CRISTAL,522,246,47.1,4,276
DI ENS,206,80,38.8,0,126
CROSSING (IRL),34,10,29.4,0,24
ETIS,153,72,47.1,0,81
GIPSA-Lab,263,179,68.1,1,84
GREYC,187,88,47.1,2,99


In [26]:
# --- Détail par employeur (vue pivotée : labo x employeur) ---
print("=== % d'auteurs avec ORCID par employeur ===")
all_employers = sorted({
    emp
    for row in rows_stats
    for emp in row["orcid_par_employeur"].keys()
})
df_emp_pivot = pd.DataFrame(
    [{
        "labo": row["labo"],
        **{emp: row["orcid_par_employeur"].get(emp, "") for emp in all_employers}
    } for row in rows_stats]
).set_index("labo")
# Ne garder que les colonnes non entièrement vides
df_emp_pivot = df_emp_pivot.loc[:, (df_emp_pivot != "").any()]
display(df_emp_pivot)

# --- Détail par fonction (vue pivotée : labo x fonction) ---
print("=== % d'auteurs avec ORCID par fonction ===")
all_roles = sorted({
    role
    for row in rows_stats
    for role in row["orcid_par_fonction"].keys()
})
df_role_pivot = pd.DataFrame(
    [{
        "labo": row["labo"],
        **{role: row["orcid_par_fonction"].get(role, "") for role in all_roles}
    } for row in rows_stats]
).set_index("labo")
df_role_pivot = df_role_pivot.loc[:, (df_role_pivot != "").any()]
display(df_role_pivot)


=== % d'auteurs avec ORCID par employeur ===


,ACOEM (France),ADEME,AGROCAMPUS OUEST,AIR&D,AMIDS Solutions,ANSES,ANSES-LSV,"AUSY, IRIT, INSA",Abdus Salam International Centre for Theoretical Physics,Académie de Montpellier,...,École des Mines de Saint-Étienne,École des Ponts ParisTech,École internationale des sciences du traitement de l'information,École nationale d'ingénieurs de Brest,École nationale des ponts et chaussées,École nationale supérieure de chimie de Paris,École nationale supérieure d’architecture Paris-Malaquais – PSL,École normale supérieure de Lyon,Électricité de France (France),ČVUT
labo,,,,,,,,,,,,,,,,,,,,,
CRAN,,,,,,,,,,,...,,,,,,,,,,
CREATIS,,,,,1/286 (0%),,,,,,...,,,,,,,,,,
CRIL,,,,,,,,,,,...,,,,,,,,,,
CRISTAL,,,,,,,,,,,...,,,,,1/522 (0%),,,,,1/522 (0%)
DI ENS,,,,,,,,,,,...,,,,,,,,,,
CROSSING (IRL),,,,,,,,,,,...,,,,,,,,,,
ETIS,,,,,,,,,,,...,,,1/153 (1%),,,,,,,
GIPSA-Lab,,,,,,,,,,,...,,,,,,,,,,
GREYC,,,,,,,,,,,...,,,,,,,,,,


=== % d'auteurs avec ORCID par fonction ===


,"""Emerite"" full professor",AI Research Engineer,AI Researcher,ASSOCIATE pROFESSOR,ATER,ATER (junior researcher),ATER (temporary teaching and research assistant),"Academic , Research Associate",Academic Visitor,Adjunct Associate Professor,...,research fellow,research fellow (chargé de recherche CNRS),researcher,"researcher, senior researcher, then ""Emerite"" full professor",senior XR developper,senior research scientist,senior researcher,surgeon,vice rector for education and research,¨Postdoctoral scholar
labo,,,,,,,,,,,,,,,,,,,,,
CRAN,,,,,,,,,,,...,,,,,,,,2/249 (1%),,
CREATIS,,1/286 (0%),,,,,,,,,...,,,,,,,,,,
CRIL,,,,,,,,,,,...,,,,,,,,,,
CRISTAL,,,,,,,,,,,...,,,,,,,,,,
DI ENS,,,,,,,,,,,...,,,,,,,,,,
CROSSING (IRL),,,,,,,,,,,...,,,,,,,,,,
ETIS,,,,,,,,,,,...,,,,,,,,,,
GIPSA-Lab,,,,1/263 (0%),,1/263 (0%),,,,,...,,,,,,,,,,
GREYC,,,,,,,,,,,...,,,,,,,,,,


In [27]:
# --- Export CSV séparés ---
df_stats[["n_auteurs", "n_avec_orcid", "pct_avec_orcid", "n_multi_orcid", "n_sans_orcid"]].to_csv(
    "labo_stats_orcid_2024.csv", encoding="utf-8-sig"
)
df_emp_pivot.to_csv("labo_stats_orcid_par_employeur_2024.csv", encoding="utf-8-sig")
df_role_pivot.to_csv("labo_stats_orcid_par_fonction_2024.csv", encoding="utf-8-sig")

print("Fichiers enregistrés :")
print("  labo_stats_orcid_2024.csv")
print("  labo_stats_orcid_par_employeur_2024.csv")
print("  labo_stats_orcid_par_fonction_2024.csv")


Fichiers enregistrés :
  labo_stats_orcid_2024.csv
  labo_stats_orcid_par_employeur_2024.csv
  labo_stats_orcid_par_fonction_2024.csv
